But : normalisation spatiale sur un landmark commun (rigide/affine). On récupère (ou dessine) un repère (ex. centre d’hétérotopie) et on aligne chaque neurone via une estimation affine (SVD + least squares).

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
import neurom as nm

def centroid(points):
    return points.mean(0)

def rigid_align(X, Y):
    # Kabsch: trouve R,t qui alignent X->Y (X,Y Nx3)
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    U,S,Vt = np.linalg.svd(Xc.T @ Yc)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0: Vt[-1,:]*=-1; R = Vt.T @ U.T
    t = Y.mean(0) - R @ X.mean(0)
    return R, t

def apply_affine(xyz, R, t):
    return (xyz @ R.T) + t

# Exemple: landmark = soma center → aligne tous les somas sur (0,0,0)
def soma_center(n):
    s = n.soma.points[:, :3]
    return centroid(s)

aligned_paths = []
for p in pd.read_csv("../data/interim/manifest.csv")["path"]:
    n = nm.load_neuron(p)
    pts = np.vstack([s.points[:,:3] for s in n.iter_sections()])  # tous points
    R,t = rigid_align(pts, pts - soma_center(n))  # ici trivial ; remplace Y par cible si tu as un atlas/landmark
    aligned = apply_affine(pts, R, t)
    # → sauvegarder coordonnées recalées pour features downstream (par id de point)
